# Etapa 14 · Verificación del paquete temporal

## Resumen
2T26: 35 cifras del trimestre y 299 con historia y comparables, procedentes de cuatro comunicados elegibles. 1T21: bloqueado por falta de prueba de versión histórica. No se calcula ni redacta análisis.

## Contexto y método
Validación local contra los archivos Bronze originales. La fecha de ingesta no acredita publicación.

### Supuestos
El corte es diario y corresponde al comunicado principal; publicaciones ajenas del mismo día se excluyen sin orden demostrable. Los archivos SEC se cotejan con su envío completo. Los datos de revisión excluidos no entran al paquete del analista.

In [1]:
from pathlib import Path
import os, sys, json, hashlib
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/analysis_agent/evidence.py').exists())
os.chdir(root);sys.path.insert(0,str(root))
from src.analysis_agent.evidence import build, validate, preserve, REVIEW
from src.config import PATHS
recent=json.loads((REVIEW/'2026Q2_review.json').read_bytes())['package']
history=json.loads((REVIEW/'2021Q1_review.json').read_bytes())['package']

## Datos y controles
Fuente: documentos SEC e IR en Bronze; tablas Silver de SEC. El expediente referencia IDs y hashes existentes. Se cotejan fechas, bytes del anexo, escala y localizadores.

In [2]:
print(validate(recent))
print(validate(history))
assert recent['coverage']['financial_thesis_possible']
assert history['readiness']=='blocked' and not history['metrics']
assert len([m for m in recent['metrics'] if m['period_id']=='2026Q2'])==35
assert all(s['version_available_at']<=recent['cutoff_date'] for s in recent['sources'])

{'status': 'passed', 'package_id': '2026Q2_808256bfb7420d221a3f2f2383c4b95d9889110ad1545fd85f24df8d442a9ff1', 'metrics': 299, 'versions': 299, 'excerpts': 157, 'sources': 4}
{'status': 'passed', 'package_id': '2021Q1_68843ff139008be3b493404c693d45f303bef76612deecaac7aac929d28a1cec', 'metrics': 0, 'versions': 0, 'excerpts': 0, 'sources': 0}


## Resultados · reproducibilidad y aislamiento
Recrear el paquete debe producir la misma huella y reutilizar la misma versión. No se escribe en Gold ni se usa su serie vigente para decidir valores.

In [3]:
def gold_hashes():
    return {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in PATHS.gold.glob('*.parquet')}
before=gold_hashes()
rebuilt=build('2026Q2')['package']
assert rebuilt==recent
first=preserve(recent);second=preserve(rebuilt)
assert first==second
assert gold_hashes()==before
print('Mismo paquete, misma versión; Gold intacto.')

Mismo paquete, misma versión; Gold intacto.


## Conclusiones
La evidencia esencial reciente está disponible. El contexto externo aún no tiene versiones elegibles en el paquete y no podrá sostener atribuciones causales. Los cálculos y conversiones se harán en Etapa 15, después de aceptación humana. No se certifican automáticamente los cortes originales de otros trimestres recientes cuyos filings llegaron después de sus comunicados.